# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

## 1. Initial Setup

In [2]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3896


In [3]:

print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 97.75 GB
MemAvailable: 978.79 GB
Free GPU Memory (GB): 39.3896

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################

Free GPU Memory (GB): 39.3896. Context: Warm up notebook.


## 2. Loading Datasets

### 2.1 WikiText

In [1]:
import os
from src.data.WikiTextDataModule import WikiTextDataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=2048, # tokenizer.model_max_length
  tokenizer_name=model_name,
  seed=3,
  n_samples=100
)

wikitext_dataloader = wikitext_data_module.test_dataloader()

print("\n################################")
print("Printing properties of WikiTextDataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(wikitext_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(wikitext_data_module.val_dataset)}")
print(f"Length of test dataset: {len(wikitext_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in wikitext_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in wikitext_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(wikitext_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



################################
Setting up WikiTextDataModule...
################################



Token indices sequence length is longer than the specified maximum sequence length for this model (6316 > 2048). Running this sequence through the model will result in indexing errors



################################
Printing properties of WikiTextDataModule...
################################

Length of train dataset: 36718
Length of validation dataset: 3760
Length of test dataset: 4358

Total number of tokens in each dataset:
Train dataset: 10892990
Validation dataset: 1142150
Test dataset: 1285622


NameError: name 'tokenizer' is not defined

In [ ]:
from datasets import load_dataset

wikitext_train_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
wikitext_val_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
wikitext_test_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

### 3.2 OpenAssistant

In [ ]:
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

oasst_data_module = OpenAssistantDataModule(
    directory_dataset=os.getcwd(),
    batch_size=1,
    sequence_length=2048,
    tokenizer_name=model_name,
    seed=1
)

oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

In [ ]:
from datasets import load_dataset

oasst_train_dataset = load_dataset("OpenAssistant/oasst1", split="train[:95%]")
oasst_val_dataset = load_dataset("OpenAssistant/oasst1", split="validation")
oasst_test_dataset = load_dataset("OpenAssistant/oasst1", split="train[95%:]")

In [43]:
from datasets import load_dataset
from src.data import C4DataModule
from transformers import AutoTokenizer

print("Loading C4 dataset")
c4_train_dataset = load_dataset(
    "allenai/c4",
    data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
    split="train"
)

print("Tokenizing dataset")
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
# tokenized_dataset = tokenizer("\n\n".join(c4_train_dataset["text"]), return_tensors="pt")

# print("Loading data module")
# c4_data_module = C4DataModule(
#     directory_dataset=os.getcwd(),
#     batch_size=1,
#     sequence_length=512,
#     tokenizer_name=model_name,
#     seed=1
# )
# tokenized_dataset = self.tokenizer("\n\n".join(dataset["text"]), return_tensors="pt")

# print("Loading dataloader")
# c4_dataloader = data_loader_from_split(c4_data_module)["train"]


Loading C4 dataset
Tokenizing dataset


In [3]:
from src.data import OpenAssistantDataModule


print("Loading Oasst dataset")
oasst_train_dataset = load_dataset(
    "OpenAssistant/oasst1",
    split="train"
)

print("Loading data module")
oasst_data_module = OpenAssistantDataModule(
    directory_dataset=os.getcwd(),
    batch_size=1,
    sequence_length=512,
    tokenizer_name=model_name,
    seed=1
)

print("Loading dataloader")
oasst_dataloader = oasst_data_module.train_dataloader()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (OpenAssistantDataModule.py, line 89)

In [14]:
from datasets import load_dataset
from transformers import AutoTokenizer
import random

def get_c4_new(nsamples, seed, seqlen, model):
    print("get_c4_new")
    traindata = load_dataset(
        'allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'}, split='train'
    )
    valdata = load_dataset(
        'allenai/c4', data_files={'validation': 'en/c4-validation.00000-of-00008.json.gz'}, split='validation'
    )

    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    
    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        while True:
            i = random.randint(0, len(traindata) - 1)
            trainenc = tokenizer(traindata[i]["text"], return_tensors="pt")
            if trainenc.input_ids.shape[1] >= seqlen:
                break
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))

    valenc = tokenizer(" ".join(valdata[:1100]["text"]), return_tensors="pt")
    valenc = valenc.input_ids[:, : (256 * seqlen)]
    return trainloader, valenc

In [15]:
from datasets import load_dataset
from transformers import AutoTokenizer
import random

def get_c4(nsamples, seed, seqlen, model):
    print("get_c4")
    traindata = load_dataset(
        'allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'}, split='train'
    )
    valdata = load_dataset(
        'allenai/c4', data_files={'validation': 'en/c4-validation.00000-of-00008.json.gz'}, split='validation'
    )


    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)

    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        while True:
            i = random.randint(0, len(traindata) - 1)
            trainenc = tokenizer(traindata[i]['text'], return_tensors='pt')
            if trainenc.input_ids.shape[1] >= seqlen:
                break
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))

    random.seed(0)
    valenc = []
    for _ in range(256):
        while True:
            i = random.randint(0, len(valdata) - 1)
            tmp = tokenizer(valdata[i]['text'], return_tensors='pt')
            if tmp.input_ids.shape[1] >= seqlen:
                break
        i = random.randint(0, tmp.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        valenc.append(tmp.input_ids[:, i:j])
    valenc = torch.hstack(valenc)

    return trainloader, valenc

In [35]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.

c4 = get_c4(100, 1, 1024, model_name)
c4_new = get_c4_new(100, 1, 1024, model_name)

get_c4


Token indices sequence length is longer than the specified maximum sequence length for this model (13835 > 2048). Running this sequence through the model will result in indexing errors


get_c4_new


Token indices sequence length is longer than the specified maximum sequence length for this model (13835 > 2048). Running this sequence through the model will result in indexing errors


In [3]:
import os
from pytorch_lightning import LightningDataModule
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from datasets import load_dataset


class TextDataset(Dataset):
    def __init__(self, dataset, tokenizer, sequence_length=2048, stride=512):
        self.tokenizer = tokenizer
        self.dataset=dataset
        self.texts = dataset["text"]
        tokenized_dataset = self.tokenizer("\n\n".join(dataset["text"]), return_tensors="pt")
        self.data = tokenized_dataset.input_ids[0, :-1]
        self.labels = tokenized_dataset.input_ids[0]
        self.sequence_length = sequence_length
        self.stride = stride

    def __len__(self):
        return len(self.data) // self.sequence_length

    def __getitem__(self, index):
        start_index = max(index * self.stride + self.stride - self.sequence_length, 0)
        end_index = start_index + self.stride
        if end_index > len(self.data):
            raise IndexError("Index out of bounds")
        input_ids = self.data[start_index:end_index]
        target_ids = self.labels[start_index + 1 : end_index + 1]
        target_ids[:-self.stride] = -100
        
        return input_ids, target_ids


class OpenAssistantDataModule(LightningDataModule):
    def __init__(self, directory_dataset=os.getcwd(), batch_size=64, sequence_length=2048, stride=512, n_lines=None, tokenizer_name=None, seed=1):
        super().__init__()
        self.directory_dataset = directory_dataset
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, legacy=False)
        self.sequence_length = sequence_length
        self.stride = stride
        self.n_lines = n_lines
        self.prepare_data()

    def prepare_data(self):
        # Load train, val, and test datasets
        # Load train, val, and test datasets
        train_split = "train[:95%]" if self.n_lines is None else f"train[:{self.n_lines}]"
        validation_split = "validation" if self.n_lines is None else f"validation[:{self.n_lines}]"
        test_split = "train[95%:]" if self.n_lines is None else f"train[{self.n_lines}:{2*self.n_lines}]"
        
        self.train_dataset = load_dataset("OpenAssistant/oasst1", split=train_split)
        self.val_dataset = load_dataset("OpenAssistant/oasst1", split=validation_split)
        self.test_dataset = load_dataset("OpenAssistant/oasst1", split=test_split)

    def train_dataloader(self, batch_size=None, sequence_length=None, stride=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        if stride is None:
            stride = self.stride
        dataset = TextDataset(self.train_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length, stride=stride)
        train_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        train_dataloader.ORIGINAL_DATASET = self.train_dataset
        return train_dataloader

    def val_dataloader(self, batch_size=None, sequence_length=None, stride=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        if stride is None:
            stride = self.stride
        dataset = TextDataset(self.val_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length, stride=stride)
        val_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        val_dataloader.ORIGINAL_DATASET = self.val_dataset
        return val_dataloader
    
    def test_dataloader(self, batch_size=None, sequence_length=None, stride=None):
        if batch_size is None:
            batch_size = self.batch_size
        if sequence_length is None:
            sequence_length = self.sequence_length
        else:
            sequence_length = min(self.sequence_length, sequence_length)
        if stride is None:
            stride = self.stride
        dataset = TextDataset(self.test_dataset, tokenizer=self.tokenizer, sequence_length=sequence_length, stride=stride)
        test_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
        test_dataloader.ORIGINAL_DATASET = self.test_dataset
        return test_dataloader

In [4]:
from src.data import OpenAssistantDataModule
from datasets import load_dataset
from transformers import AutoTokenizer
import random

print("Loading Oasst dataset")
oasst_train_dataset = load_dataset(
    "OpenAssistant/oasst1",
    split="train"
)

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.

print("Loading data module")
oasst_data_module = OpenAssistantDataModule(
    directory_dataset=os.getcwd(),
    batch_size=1,
    sequence_length=512,
    n_samples=100,
    tokenizer_name=model_name,
    seed=1
)

print("Loading dataloader")
oasst_dataloader = oasst_data_module.train_dataloader()

Loading Oasst dataset
Loading data module
Loading dataloader


Token indices sequence length is longer than the specified maximum sequence length for this model (16912 > 2048). Running this sequence through the model will result in indexing errors


In [ ]:
for i in range(5):
    print(oasst_dataloader.dataset[i])

In [6]:
oasst_dataloader.dataset.texts

['Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.',
 '"Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees. The presence of a monopsony can result in lower wages and reduced employment opportunities for workers, as the employer has little incentive to increase wages or provide better working conditions.\n\nRecent research has identified potential monopsonies in industries such as retail and fast food, where a few large companies control a significant portion of the market (Bivens & Mishel, 2013). In these industries, workers often face low wages, limited benefits, and reduced bargaining power, leading to a situation w

In [5]:
ptb_train_dataset = load_dataset('ptb_text_only', 'penn_treebank', split="train", trust_remote_code=True)

In [7]:
c4_dataloader = data_loader_from_split(c4_train_dataset)['train']

AttributeError: 'Dataset' object has no attribute 'train_dataloader'

In [9]:
ptb_train_dataset

Dataset({
    features: ['sentence'],
    num_rows: 42068
})

In [10]:
ptb_train_dataset['sentence']

['aer banknote berlitz calloway centrust cluett fromstein gitano guterman hydro-quebec ipo kia memotec mlx nahb punts rake regatta rubens sim snack-food ssangyong swapo wachter',
 'pierre <unk> N years old will join the board as a nonexecutive director nov. N',
 'mr. <unk> is chairman of <unk> n.v. the dutch publishing group',
 'rudolph <unk> N years old and former chairman of consolidated gold fields plc was named a nonexecutive director of this british industrial conglomerate',
 'a form of asbestos once used to make kent cigarette filters has caused a high percentage of cancer deaths among a group of workers exposed to it more than N years ago researchers reported',
 'the asbestos fiber <unk> is unusually <unk> once it enters the <unk> with even brief exposures to it causing symptoms that show up decades later researchers said',
 '<unk> inc. the unit of new york-based <unk> corp. that makes kent cigarettes stopped using <unk> in its <unk> cigarette filters in N',
 "although prelimina